# Unidad 5 · Bloque 1: Clustering
**Métodos de Análisis de Datos I — UNS**

---

## Contenido
1. Métricas de distancia
2. K-means
3. Clustering jerárquico
4. DBSCAN
5. Modelos de Mezclas Gaussianas (GMM)
6. Evaluación de clustering

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
from sklearn.datasets import make_blobs, make_moons, load_iris
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans, AgglomerativeClustering, DBSCAN
from sklearn.mixture import GaussianMixture
from sklearn.metrics import silhouette_score, davies_bouldin_score, calinski_harabasz_score
from scipy.cluster.hierarchy import dendrogram, linkage
from scipy.spatial.distance import cdist

np.random.seed(42)
print('Librerías cargadas correctamente.')

---
## 1. Métricas de distancia

### Definiciones clave

> **Distancia euclidiana:** $d_E(\mathbf{x}, \mathbf{z}) = \sqrt{\sum_{j=1}^p (x_j - z_j)^2}$  
> La más común. Mide la distancia en línea recta. Sensible a la escala de las variables.

> **Distancia Manhattan:** $d_M(\mathbf{x}, \mathbf{z}) = \sum_{j=1}^p |x_j - z_j|$  
> Suma de diferencias absolutas. Más robusta a outliers que la euclidiana.

> **Distancia coseno:** $d_{\cos}(\mathbf{x}, \mathbf{z}) = 1 - \frac{\mathbf{x} \cdot \mathbf{z}}{\|\mathbf{x}\|\|\mathbf{z}\|}$  
> Mide el ángulo entre dos vectores. Ideal para datos de texto donde importa la dirección, no la magnitud.

**Regla de oro:** siempre estandarizar (media 0, desvío 1) antes de calcular distancias euclidianas o de Manhattan, para que ninguna variable domine por su escala.

In [ ]:
# Tres puntos en R^2
A = np.array([1.0, 2.0])
B = np.array([4.0, 6.0])
C = np.array([10.0, 0.5])

def dist_euclidiana(x, z):
    return np.sqrt(np.sum((x - z)**2))

def dist_manhattan(x, z):
    return np.sum(np.abs(x - z))

def dist_coseno(x, z):
    return 1 - np.dot(x, z) / (np.linalg.norm(x) * np.linalg.norm(z))

print('=== Distancias entre A y B ===')
print(f'  Euclidiana : {dist_euclidiana(A, B):.4f}')
print(f'  Manhattan  : {dist_manhattan(A, B):.4f}')
print(f'  Coseno     : {dist_coseno(A, B):.4f}')

print('\n=== Distancias entre A y C ===')
print(f'  Euclidiana : {dist_euclidiana(A, C):.4f}')
print(f'  Manhattan  : {dist_manhattan(A, C):.4f}')
print(f'  Coseno     : {dist_coseno(A, C):.4f}')

In [ ]:
# Visualización: impacto del escalado en las distancias
# Variable 1: edad (0-80), Variable 2: ingreso (0-100.000)
datos_sin_escalar = np.array([[30, 20000], [35, 80000], [60, 25000]])
scaler = StandardScaler()
datos_escalados = scaler.fit_transform(datos_sin_escalar)

nombres = ['Persona A', 'Persona B', 'Persona C']

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

for ax, datos, titulo in zip(axes,
                              [datos_sin_escalar, datos_escalados],
                              ['Sin escalar (domina el ingreso)', 'Escalado (ambas variables pesan igual)']):
    for i, (p, n) in enumerate(zip(datos, nombres)):
        ax.scatter(*p, s=120, zorder=5)
        ax.annotate(n, p, textcoords='offset points', xytext=(8, 4), fontsize=9)
    ax.set_title(titulo, fontsize=11)
    ax.grid(True, alpha=0.3)

axes[0].set_xlabel('Edad'); axes[0].set_ylabel('Ingreso')
axes[1].set_xlabel('Edad (std)'); axes[1].set_ylabel('Ingreso (std)')
plt.suptitle('Efecto del escalado en la geometría de los datos', fontsize=12, fontweight='bold')
plt.tight_layout()
plt.show()

---
## 2. K-means

### Definiciones clave

> **Inercia (within-cluster sum of squares):** $W = \sum_{l=1}^k \sum_{\mathbf{x}_i \in C_l} \|\mathbf{x}_i - \boldsymbol{\mu}_l\|^2$  
> Es lo que K-means minimiza. Mide cuán compactos son los clusters. Siempre decrece al aumentar $k$, por eso no se puede usar directamente para elegir $k$.

> **Centroide:** $\boldsymbol{\mu}_l = \frac{1}{|C_l|} \sum_{\mathbf{x}_i \in C_l} \mathbf{x}_i$  
> El "centro de masa" de cada cluster. K-means lo actualiza en cada iteración.

> **K-means++:** estrategia de inicialización que elige los centroides iniciales con probabilidad proporcional a su distancia al centroide ya elegido más cercano. Reduce drásticamente el riesgo de quedar atrapado en mínimos locales malos.

In [ ]:
# Dataset: 3 grupos bien separados (sintético)
X, y_true = make_blobs(n_samples=300, centers=3, cluster_std=0.8, random_state=42)
X = StandardScaler().fit_transform(X)

# Ajustar K-means con k=3
km = KMeans(n_clusters=3, init='k-means++', n_init=10, random_state=42)
etiquetas = km.fit_predict(X)
centroides = km.cluster_centers_

fig, axes = plt.subplots(1, 2, figsize=(13, 5))

# Datos originales (sin color)
axes[0].scatter(X[:, 0], X[:, 1], c='gray', alpha=0.5, s=30)
axes[0].set_title('Datos originales (sin etiquetas)', fontsize=11)
axes[0].grid(True, alpha=0.3)

# Resultado K-means
colores = ['#E74C3C', '#2ECC71', '#3498DB']
for l in range(3):
    mask = etiquetas == l
    axes[1].scatter(X[mask, 0], X[mask, 1], c=colores[l], alpha=0.6, s=30, label=f'Cluster {l+1}')
axes[1].scatter(centroides[:, 0], centroides[:, 1], c='black', marker='X', s=200, zorder=5, label='Centroides')
axes[1].set_title(f'K-means (k=3) — Inercia: {km.inertia_:.2f}', fontsize=11)
axes[1].legend(fontsize=9)
axes[1].grid(True, alpha=0.3)

plt.suptitle('K-means sobre datos sintéticos', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
# Método del codo + coeficiente de silueta para elegir k
k_vals = range(2, 10)
inercias = []
siluetas = []

for k in k_vals:
    km_k = KMeans(n_clusters=k, init='k-means++', n_init=10, random_state=42)
    lab = km_k.fit_predict(X)
    inercias.append(km_k.inertia_)
    siluetas.append(silhouette_score(X, lab))

fig, axes = plt.subplots(1, 2, figsize=(13, 4))

axes[0].plot(list(k_vals), inercias, 'o-', color='steelblue', linewidth=2)
axes[0].axvline(3, color='red', linestyle='--', alpha=0.7, label='k=3 (codo)')
axes[0].set_xlabel('Número de clusters k')
axes[0].set_ylabel('Inercia')
axes[0].set_title('Método del codo', fontsize=11)
axes[0].legend()
axes[0].grid(True, alpha=0.3)

axes[1].plot(list(k_vals), siluetas, 's-', color='darkorange', linewidth=2)
axes[1].axvline(3, color='red', linestyle='--', alpha=0.7, label='k=3 (máximo)')
axes[1].set_xlabel('Número de clusters k')
axes[1].set_ylabel('Coeficiente de silueta')
axes[1].set_title('Coeficiente de silueta', fontsize=11)
axes[1].legend()
axes[1].grid(True, alpha=0.3)

plt.suptitle('Selección de k: codo y silueta', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

k_optimo = list(k_vals)[np.argmax(siluetas)]
print(f'k óptimo según silueta: {k_optimo} (silueta = {max(siluetas):.4f})')

In [ ]:
# Limitación de K-means: falla con formas no esféricas
X_lunas, _ = make_moons(n_samples=300, noise=0.05, random_state=42)
X_lunas = StandardScaler().fit_transform(X_lunas)

km_lunas = KMeans(n_clusters=2, random_state=42)
etiquetas_lunas = km_lunas.fit_predict(X_lunas)

fig, axes = plt.subplots(1, 2, figsize=(13, 4))

axes[0].scatter(X_lunas[:, 0], X_lunas[:, 1], c='gray', alpha=0.5, s=30)
axes[0].set_title('Datos en forma de lunas (estructura real)', fontsize=11)
axes[0].grid(True, alpha=0.3)

for l in range(2):
    mask = etiquetas_lunas == l
    axes[1].scatter(X_lunas[mask, 0], X_lunas[mask, 1], c=colores[l], alpha=0.6, s=30, label=f'Cluster {l+1}')
axes[1].scatter(km_lunas.cluster_centers_[:, 0], km_lunas.cluster_centers_[:, 1],
                c='black', marker='X', s=200, zorder=5)
axes[1].set_title('K-means falla: asume forma esférica', fontsize=11)
axes[1].legend(fontsize=9)
axes[1].grid(True, alpha=0.3)

plt.suptitle('Limitación de K-means con clusters no esféricos', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()
print('→ Para este caso, DBSCAN es el método adecuado (ver sección 4).')

---
## 3. Clustering jerárquico

### Definiciones clave

> **Dendrograma:** árbol que representa la jerarquía de fusiones. La altura de cada unión indica la distancia (disimilitud) a la que ocurrió. Cortando el dendrograma a una altura dada se obtienen $k$ clusters.

> **Enlace de Ward:** fusiona los dos clusters cuya unión produce el menor aumento de inercia total. Es el criterio más recomendado para datos continuos, ya que tiende a producir clusters de tamaño similar.

> **Ventaja clave frente a K-means:** no requiere especificar $k$ de antemano. El analista elige el corte a posteriori observando el dendrograma.

In [ ]:
# Dataset: iris (150 flores, 4 variables, 3 especies)
iris = load_iris()
X_iris = StandardScaler().fit_transform(iris.data)
especies = iris.target_names

# Dendrograma con enlace Ward
Z = linkage(X_iris, method='ward')

fig, ax = plt.subplots(figsize=(14, 5))
dendrogram(Z, ax=ax, truncate_mode='lastp', p=30,
           leaf_rotation=90, leaf_font_size=8,
           color_threshold=5.0)
ax.axhline(y=5.0, color='red', linestyle='--', alpha=0.7, label='Corte en altura=5 → 3 clusters')
ax.set_title('Dendrograma — Dataset Iris · Enlace Ward', fontsize=13, fontweight='bold')
ax.set_xlabel('Observaciones (agrupadas)')
ax.set_ylabel('Distancia de fusión')
ax.legend(fontsize=9)
plt.tight_layout()
plt.show()

In [ ]:
# Comparar 4 métodos de enlace
metodos = ['ward', 'complete', 'average', 'single']
nombres_metodos = ['Ward', 'Completo', 'Promedio', 'Simple']

fig, axes = plt.subplots(2, 2, figsize=(14, 8))
axes = axes.flatten()

for ax, metodo, nombre in zip(axes, metodos, nombres_metodos):
    Z_m = linkage(X_iris, method=metodo)
    dendrogram(Z_m, ax=ax, truncate_mode='lastp', p=20,
               leaf_rotation=90, leaf_font_size=7, no_labels=True)
    ax.set_title(f'Enlace {nombre}', fontsize=11)
    ax.set_ylabel('Distancia')

plt.suptitle('Comparación de métodos de enlace — Iris', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

# Asignar clusters con Ward y k=3
hc = AgglomerativeClustering(n_clusters=3, linkage='ward')
etiquetas_hc = hc.fit_predict(X_iris)
sil_hc = silhouette_score(X_iris, etiquetas_hc)
print(f'Clustering jerárquico (Ward, k=3) — Silueta: {sil_hc:.4f}')

---
## 4. DBSCAN

### Definiciones clave

> **Punto núcleo:** $\mathbf{x}_i$ es núcleo si tiene al menos $m$ vecinos dentro de un radio $\varepsilon$ (incluyéndose a sí mismo).

> **Punto borde:** no es núcleo, pero cae dentro de la vecindad $\varepsilon$ de algún punto núcleo.

> **Ruido (noise):** ni núcleo ni borde. DBSCAN los etiqueta con $-1$. Son los **outliers** explícitos del algoritmo.

> **Parámetros clave:**  
> - $\varepsilon$ (`eps`): radio de vecindad. Valores típicos: explorar con gráfico de k-distancias.  
> - $m$ (`min_samples`): mínimo de puntos para ser núcleo. Regla práctica: $m \geq \text{dim} + 1$.

In [ ]:
# DBSCAN sobre datos en forma de lunas: el caso donde K-means falló
db = DBSCAN(eps=0.3, min_samples=5)
etiquetas_db = db.fit_predict(X_lunas)

n_clusters = len(set(etiquetas_db)) - (1 if -1 in etiquetas_db else 0)
n_ruido = np.sum(etiquetas_db == -1)

fig, axes = plt.subplots(1, 2, figsize=(13, 4))

# K-means (malo)
for l in range(2):
    mask = etiquetas_lunas == l
    axes[0].scatter(X_lunas[mask, 0], X_lunas[mask, 1], c=colores[l], alpha=0.5, s=30)
axes[0].set_title('K-means (resultado incorrecto)', fontsize=11)
axes[0].grid(True, alpha=0.3)

# DBSCAN (correcto)
colores_db = {0: '#E74C3C', 1: '#2ECC71', -1: 'gray'}
for lab in set(etiquetas_db):
    mask = etiquetas_db == lab
    label = f'Cluster {lab+1}' if lab >= 0 else 'Ruido'
    axes[1].scatter(X_lunas[mask, 0], X_lunas[mask, 1],
                    c=colores_db.get(lab, 'purple'), alpha=0.6 if lab >= 0 else 0.3,
                    s=30 if lab >= 0 else 15, label=label)
axes[1].set_title(f'DBSCAN — {n_clusters} clusters, {n_ruido} puntos de ruido', fontsize=11)
axes[1].legend(fontsize=9)
axes[1].grid(True, alpha=0.3)

plt.suptitle('DBSCAN detecta clusters de forma arbitraria', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
# Sensibilidad de DBSCAN a los parámetros eps y min_samples
configs = [(0.15, 5), (0.3, 5), (0.5, 5), (0.3, 15)]
titulos = ['eps=0.15, min=5\n(muy estricto)', 'eps=0.30, min=5\n(óptimo)',
           'eps=0.50, min=5\n(muy permisivo)', 'eps=0.30, min=15\n(núcleo exigente)']

fig, axes = plt.subplots(1, 4, figsize=(16, 4))

for ax, (eps, ms), titulo in zip(axes, configs, titulos):
    db_c = DBSCAN(eps=eps, min_samples=ms)
    lab_c = db_c.fit_predict(X_lunas)
    n_c = len(set(lab_c)) - (1 if -1 in lab_c else 0)
    n_r = np.sum(lab_c == -1)
    unique_labs = sorted(set(lab_c))
    cmap = plt.cm.get_cmap('tab10', max(len(unique_labs), 2))
    for i, lab in enumerate(unique_labs):
        mask = lab_c == lab
        color = 'lightgray' if lab == -1 else cmap(i)
        ax.scatter(X_lunas[mask, 0], X_lunas[mask, 1], c=[color]*mask.sum(),
                   alpha=0.5 if lab == -1 else 0.7, s=20)
    ax.set_title(f'{titulo}\n{n_c} clusters, {n_r} ruido', fontsize=8)
    ax.grid(True, alpha=0.3)
    ax.set_xticks([]); ax.set_yticks([])

plt.suptitle('Sensibilidad de DBSCAN a los parámetros', fontsize=12, fontweight='bold')
plt.tight_layout()
plt.show()

---
## 5. Modelos de Mezclas Gaussianas (GMM)

### Definiciones clave

> **GMM (Gaussian Mixture Model):** asume que los datos fueron generados por una mezcla de $k$ distribuciones gaussianas:  
> $$p(\mathbf{x}) = \sum_{l=1}^k \pi_l \, \mathcal{N}(\mathbf{x} \mid \boldsymbol{\mu}_l, \boldsymbol{\Sigma}_l)$$

> **Responsabilidad** $r_{il}$: probabilidad a posteriori de que la observación $i$ haya sido generada por el componente $l$. Es la "asignación blanda" que diferencia GMM de K-means.

> **Algoritmo EM (Expectation-Maximization):** alterna entre el paso E (calcular responsabilidades) y el paso M (actualizar parámetros maximizando la verosimilitud esperada).

> **BIC (Bayesian Information Criterion):** $\text{BIC} = -2\ln(\hat{L}) + p \ln(n)$, donde $p$ es el número de parámetros. Se usa para elegir $k$. Menor BIC = mejor modelo.

In [ ]:
# Dataset: dos grupos con formas elípticas distintas (GMM brilla aquí)
np.random.seed(42)
X1 = np.random.multivariate_normal([0, 0], [[1, 0.8], [0.8, 1]], 200)
X2 = np.random.multivariate_normal([4, 2], [[2, -0.5], [-0.5, 0.5]], 150)
X_gmm = np.vstack([X1, X2])
X_gmm = StandardScaler().fit_transform(X_gmm)

gmm = GaussianMixture(n_components=2, covariance_type='full', random_state=42)
gmm.fit(X_gmm)
etiquetas_gmm = gmm.predict(X_gmm)
probs_gmm = gmm.predict_proba(X_gmm)  # asignación blanda

fig, axes = plt.subplots(1, 2, figsize=(13, 5))

# Clusters GMM
for l in range(2):
    mask = etiquetas_gmm == l
    axes[0].scatter(X_gmm[mask, 0], X_gmm[mask, 1], c=colores[l], alpha=0.5, s=30, label=f'Componente {l+1}')
axes[0].set_title('GMM — Asignación dura (argmax)', fontsize=11)
axes[0].legend(fontsize=9)
axes[0].grid(True, alpha=0.3)

# Probabilidad de pertenencia al componente 1 (asignación blanda)
sc = axes[1].scatter(X_gmm[:, 0], X_gmm[:, 1], c=probs_gmm[:, 0],
                     cmap='RdYlGn', alpha=0.7, s=30, vmin=0, vmax=1)
plt.colorbar(sc, ax=axes[1], label='P(componente 1)')
axes[1].set_title('GMM — Probabilidad de pertenencia (asignación blanda)', fontsize=11)
axes[1].grid(True, alpha=0.3)

plt.suptitle('GMM: asignación dura vs. blanda', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

print(f'Log-verosimilitud: {gmm.score(X_gmm)*len(X_gmm):.2f}')
print(f'BIC: {gmm.bic(X_gmm):.2f}')

In [ ]:
# Selección de número de componentes con BIC y AIC
componentes = range(1, 8)
bic_vals = []
aic_vals = []

for k in componentes:
    gm = GaussianMixture(n_components=k, covariance_type='full', random_state=42)
    gm.fit(X_gmm)
    bic_vals.append(gm.bic(X_gmm))
    aic_vals.append(gm.aic(X_gmm))

fig, ax = plt.subplots(figsize=(9, 4))
ax.plot(list(componentes), bic_vals, 'o-', color='steelblue', linewidth=2, label='BIC')
ax.plot(list(componentes), aic_vals, 's--', color='darkorange', linewidth=2, label='AIC')
ax.axvline(2, color='red', linestyle=':', alpha=0.7, label='k=2 (mínimo BIC)')
ax.set_xlabel('Número de componentes')
ax.set_ylabel('Criterio de información')
ax.set_title('Selección de k en GMM: BIC y AIC', fontsize=12, fontweight='bold')
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

k_bic = list(componentes)[np.argmin(bic_vals)]
print(f'k óptimo según BIC: {k_bic}')

---
## 6. Evaluación de clustering

### Definiciones clave

> **Coeficiente de silueta** $\bar{s} \in [-1, 1]$: para cada punto $i$, $s_i = (b_i - a_i)/\max(a_i, b_i)$, donde $a_i$ es la distancia media intra-cluster y $b_i$ la distancia media al cluster más cercano. **Mayor es mejor.** Valores $> 0.5$ indican estructura clara.

> **Índice de Davies-Bouldin:** promedio del cociente entre dispersión intra-cluster y separación inter-cluster. **Menor es mejor.**

> **Índice de Calinski-Harabasz:** cociente entre dispersión inter-cluster e intra-cluster (análogo al estadístico $F$ de ANOVA). **Mayor es mejor.**

> **Regla práctica:** ninguna métrica es perfecta. Usarlas en conjunto y validar siempre con criterio de dominio.

In [ ]:
# Comparar K-means, Jerárquico, DBSCAN y GMM sobre el dataset de blobs
X_eval, _ = make_blobs(n_samples=300, centers=3, cluster_std=1.0, random_state=42)
X_eval = StandardScaler().fit_transform(X_eval)

modelos = {
    'K-means (k=3)':         KMeans(n_clusters=3, random_state=42, n_init=10).fit_predict(X_eval),
    'Jerárquico (Ward, k=3)': AgglomerativeClustering(n_clusters=3, linkage='ward').fit_predict(X_eval),
    'GMM (k=3)':              GaussianMixture(n_components=3, random_state=42).fit(X_eval).predict(X_eval),
}

print(f"{'Método':<28} {'Silueta':>10} {'Davies-Bouldin':>16} {'Calinski-Harabasz':>20}")
print('-' * 76)
for nombre, etiq in modelos.items():
    sil  = silhouette_score(X_eval, etiq)
    db   = davies_bouldin_score(X_eval, etiq)
    ch   = calinski_harabasz_score(X_eval, etiq)
    print(f'{nombre:<28} {sil:>10.4f} {db:>16.4f} {ch:>20.2f}')

print('\nInterpretación: Silueta ↑ mejor | Davies-Bouldin ↓ mejor | Calinski-Harabasz ↑ mejor')

In [ ]:
# Diagrama de silueta para K-means k=3
from sklearn.metrics import silhouette_samples

km3 = KMeans(n_clusters=3, random_state=42, n_init=10)
etiq3 = km3.fit_predict(X_eval)
sil_vals = silhouette_samples(X_eval, etiq3)

fig, ax = plt.subplots(figsize=(9, 5))
y_lower = 10
for l in range(3):
    sil_l = np.sort(sil_vals[etiq3 == l])
    size_l = len(sil_l)
    y_upper = y_lower + size_l
    ax.fill_betweenx(np.arange(y_lower, y_upper), 0, sil_l,
                     alpha=0.7, color=colores[l], label=f'Cluster {l+1}')
    ax.text(-0.05, y_lower + 0.5 * size_l, str(l+1), fontsize=10, va='center')
    y_lower = y_upper + 10

ax.axvline(silhouette_score(X_eval, etiq3), color='red', linestyle='--',
           label=f'Silueta media = {silhouette_score(X_eval, etiq3):.3f}')
ax.set_xlabel('Coeficiente de silueta')
ax.set_ylabel('Cluster')
ax.set_title('Diagrama de silueta — K-means (k=3)', fontsize=12, fontweight='bold')
ax.legend(fontsize=9)
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

---
## Resumen del bloque

| Método | Requiere $k$ | Forma de clusters | Outliers | Asignación |
|--------|-------------|-------------------|----------|------------|
| K-means | Sí | Esférica | Sensible | Dura |
| Jerárquico | No* | Flexible | Sensible | Dura |
| DBSCAN | No | Arbitraria | Detecta | Dura |
| GMM | Sí | Elíptica | Moderado | Blanda |

*elige $k$ al cortar el dendrograma.

**Pipeline recomendado:**
1. Estandarizar las variables.
2. Explorar con K-means + codo + silueta.
3. Refinar con DBSCAN si hay ruido o forma irregular.
4. Usar GMM si se necesitan probabilidades de pertenencia.
5. Evaluar con múltiples métricas y validar con criterio de dominio.